> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `1005795510073716` · task `raw_to_bronze` · notebook `01_raw_to_bronze` · started 2026-11-17 20:28 UTC · result **SUCCESS**  
> Parameters: `batch_ids={{job.parameters.batch_ids}}`, `catalog={{job.parameters.catalog}}`, `drift_mode={{job.parameters.drift_mode}}`, `end_date={{job.parameters.end_date}}`, `landing_path={{job.parameters.landing_path}}`, `load_type={{job.parameters.load_type}}`, `mode={{job.parameters.mode}}`, `start_date={{job.parameters.start_date}}`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 01 · Raw → Bronze
Loads landing batches (JSONL + `_manifest.json`) into the Bronze Delta tables using **explicit StructType
schemas** (no inference). Invalid records go to `so_bronze.quarantine_records`, schema drift is recorded in
`so_ops.schema_drift_events`, and every file processed writes a row to `so_ops.pipeline_execution_logs`.

| Widget | Standard incremental run | Backfill |
|---|---|---|
| `mode` | `incremental`: every batch not yet loaded | `backfill`: reload the selected batches |
| `load_type` | `all` | `all`, `full` or `incremental` |
| `batch_ids` | empty | e.g. `incr_20260925T155859Z,full_2025-01-01_2025-04-01` |
| `start_date` / `end_date` | empty | extraction-date range, `YYYY-MM-DD`, inclusive |
| `landing_path` | empty | an explicit batch folder, e.g. `/Volumes/workspace/so_raw/landing/incremental/incr_...` |
| `drift_mode` | `evolve` | `evolve` adds new columns (mergeSchema); `rescue` keeps them in `_rescued_data` |

In [1]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.dropdown("load_type", "all", ["all", "full", "incremental"])
dbutils.widgets.text("batch_ids", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("landing_path", "")
dbutils.widgets.dropdown("drift_mode", "evolve", ["evolve", "rescue"])

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
from so_lakehouse.bronze import run_raw_to_bronze

cfg = build_config()
audit = run_raw_to_bronze(
    spark, cfg,
    mode=widget("mode"),
    load_type=widget("load_type"),
    batch_ids=widget("batch_ids") or None,
    start_date=widget("start_date") or None,
    end_date=widget("end_date") or None,
    landing_path=widget("landing_path") or None,
    fail_on_error=True,
)

Raw-to-Bronze: 7 batch(es) selected, mode=incremental
[SKIPPED] Raw-to-Bronze | *                                  | read=0 ins=0 upd=0 del=0 quar=0 | 0.0s | already loaded; use mode=backfill to reload
[SKIPPED] Raw-to-Bronze | *                                  | read=0 ins=0 upd=0 del=0 quar=0 | 0.0s | already loaded; use mode=backfill to reload
[SKIPPED] Raw-to-Bronze | *                                  | read=0 ins=0 upd=0 del=0 quar=0 | 0.0s | already loaded; use mode=backfill to reload
[SKIPPED] Raw-to-Bronze | *                                  | read=0 ins=0 upd=0 del=0 quar=0 | 0.0s | already loaded; use mode=backfill to reload
[SKIPPED] Raw-to-Bronze | *                                  | read=0 ins=0 upd=0 del=0 quar=0 | 0.0s | already loaded; use mode=backfill to reload
[SKIPPED] Raw-to-Bronze | *                                  | read=0 ins=0 upd=0 del=0 quar=0 | 0.0s | already loaded; use mode=backfill to reload
[SUCCESS] Raw-to-Bronze | questions                       

In [4]:
display(spark.table(cfg.table("ops.pipeline_execution_logs"))
        .where(f"run_id = '{audit.run_id}'").orderBy("start_time"))

log_id,run_id,pipeline_name,layer,run_mode,load_type,entity,batch_id,source_parameter,target_table,start_time,end_time,duration_seconds,status,rows_read,rows_inserted,rows_updated,rows_deleted,rows_quarantined,message,load_timestamp
892de01f-ad69-4117-937c-2f5af310c270,raw_to_bronze-20261117T202841Z-c4f031,raw_to_bronze,Raw-to-Bronze,incremental,full,*,full_2025-04-01_2025-07-01,/Volumes/workspace/so_raw/landing/full/full_2025-04-01_2025-07-01,-,2026-11-17T20:29:19.543Z,2026-11-17T20:29:19.543Z,3.5e-05,SKIPPED,0,0,0,0,0,already loaded; use mode=backfill to reload,2026-11-17T20:29:19.543Z
b22d588a-2104-4888-a355-50258d9c7f5d,raw_to_bronze-20261117T202841Z-c4f031,raw_to_bronze,Raw-to-Bronze,incremental,full,*,full_2025-01-01_2025-04-01,/Volumes/workspace/so_raw/landing/full/full_2025-01-01_2025-04-01,-,2026-11-17T20:29:35.112Z,2026-11-17T20:29:35.112Z,3.4e-05,SKIPPED,0,0,0,0,0,already loaded; use mode=backfill to reload,2026-11-17T20:29:35.112Z
8102bd9e-b7ef-406a-9d37-a2a6c376648c,raw_to_bronze-20261117T202841Z-c4f031,raw_to_bronze,Raw-to-Bronze,incremental,incremental,*,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z,-,2026-11-17T20:29:40.157Z,2026-11-17T20:29:40.157Z,3.1e-05,SKIPPED,0,0,0,0,0,already loaded; use mode=backfill to reload,2026-11-17T20:29:40.158Z
45593c6e-d4da-4a7e-82cd-0affef6a37aa,raw_to_bronze-20261117T202841Z-c4f031,raw_to_bronze,Raw-to-Bronze,incremental,incremental,*,incr_20261007T155914Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261007T155914Z,-,2026-11-17T20:29:46.141Z,2026-11-17T20:29:46.141Z,3.2e-05,SKIPPED,0,0,0,0,0,already loaded; use mode=backfill to reload,2026-11-17T20:29:46.141Z
bc71d456-1caf-4ba2-a50a-627b22a8224b,raw_to_bronze-20261117T202841Z-c4f031,raw_to_bronze,Raw-to-Bronze,incremental,incremental,*,incr_20261007T160033Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261007T160033Z,-,2026-11-17T20:29:54.325Z,2026-11-17T20:29:54.325Z,2.9e-05,SKIPPED,0,0,0,0,0,already loaded; use mode=backfill to reload,2026-11-17T20:29:54.325Z
dba2d4e6-89a8-49ca-b320-7bcbd95267a6,raw_to_bronze-20261117T202841Z-c4f031,raw_to_bronze,Raw-to-Bronze,incremental,incremental,*,incr_20261008T000000Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261008T000000Z,-,2026-11-17T20:29:59.205Z,2026-11-17T20:29:59.205Z,2.7e-05,SKIPPED,0,0,0,0,0,already loaded; use mode=backfill to reload,2026-11-17T20:29:59.205Z
2941c854-4893-48af-bc87-489f18815c73,raw_to_bronze-20261117T202841Z-c4f031,raw_to_bronze,Raw-to-Bronze,incremental,incremental,questions,incr_20261117T202706Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261117T202706Z/questions.jsonl,workspace.so_bronze.questions,2026-11-17T20:30:04.355Z,2026-11-17T20:30:46.473Z,42.118473,SUCCESS,33,33,0,0,0,,2026-11-17T20:30:46.473Z
f19ae8c2-b7c1-4a66-9808-8c0b69972a2a,raw_to_bronze-20261117T202841Z-c4f031,raw_to_bronze,Raw-to-Bronze,incremental,incremental,answers,incr_20261117T202706Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261117T202706Z/answers.jsonl,workspace.so_bronze.answers,2026-11-17T20:30:49.765Z,2026-11-17T20:31:06.611Z,16.846006,SUCCESS,49,49,0,0,0,,2026-11-17T20:31:06.611Z
60d85a85-ef92-4960-8c20-b93da2d8276b,raw_to_bronze-20261117T202841Z-c4f031,raw_to_bronze,Raw-to-Bronze,incremental,incremental,deleted_questions,incr_20261117T202706Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261117T202706Z/deleted_questions.jsonl,workspace.so_bronze.deleted_questions,2026-11-17T20:31:09.586Z,2026-11-17T20:31:21.574Z,11.988169,SUCCESS,0,0,0,0,0,,2026-11-17T20:31:21.574Z
73120bd7-da01-4434-9945-4ad4238c14cf,raw_to_bronze-20261117T202841Z-c4f031,raw_to_bronze,Raw-to-Bronze,incremental,incremental,question_snapshots,incr_20261117T202706Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261117T202706Z/question_snapshots.jsonl,workspace.so_bronze.question_snapshots,2026-11-17T20:31:25.031Z,2026-11-17T20:31:36.518Z,11.487724,SUCCESS,0,0,0,0,0,,2026-11-17T20:31:36.518Z
